# Soc5044 Computational Sociology W11 Practicum - Text Analysis Part 1

These teaching materials are adpated from [*Introduction to Cultural Analytics & Python*](https://melaniewalsh.github.io/Intro-Cultural-Analytics/welcome.html) (Walsh 2021) for the course "Soc5044 Computational Sociology" lectured at the sociology department of National Taiwan University by Dr. Chen-Shuo Hong.

In particular, this file uses the following chapters:
- "Term-Frequency Inverse Document Frequency - With HathiTrust", Chapter 5 Text Analysis
- "Term-Frequency Inverse Document Frequency - With Scikit-Learn", Chapter 5 Text Analysis

> Melanie Walsh, Introduction to Cultural Analytics & Python, Version 1 (2021), https://doi.org/10.5281/zenodo.4411250.

Editor: Jun-Wei Liu (b09902009@ntu.edu.tw)

---

## TF-IDF

In the next lessons, we're going to learn about a text analysis method called *term frequency–inverse document frequency*, often abbreviated *tf-idf*.

While calculating the most frequent words in a text can be useful, the most frequent words in a text usually aren't the most interesting words in a text, even if we get rid of stop words ("the, "and," "to," etc.). Tf-idf is a method that builds off word frequency but it more specifically tries to identify the most distinctively frequent and significant words. 

If you already have a collection of plain text (.txt) files that you'd like to analyze, one of the easiest ways to calculate tf-idf scores is to use the Python library scikit-learn. It has a quick and nifty module called [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html), which does all the math for you behind the scenes.

We will also cover how to calculate tf-idf scores for in-copyright texts using extracted features from the [HathiTrust Digital Library](https://www.hathitrust.org/), which contains digitized books from Google Books as well as many university libraries.

# 1. TF-IDF with HathiTrust Data

In this lesson, we're going to learn about a text analysis method called *term frequency–inverse document frequency*, often abbreviated *tf-idf*.

While calculating the most frequent words in a text can be useful, the most frequent words in a text usually aren't the most interesting words in a text, even if we get rid of stop words ("the, "and," "to," etc.). Tf-idf is a method that builds off word frequency but it more specifically tries to identify the most distinctively frequent or significant words in a document. 

In this lesson, we will cover how to:
- Calculate and normalize tf-idf scores for each short story in Edward P. Jones's *Lost in the City*
- Download and process HathiTrust extracted features — that is, word frequencies for books in the HathiTrust Digital Library (including in-copyright books like *Lost in the City*)
- Prepare HathiTrust extracted features for tf-idf analysis

## Dataset

### *Lost in the City* by Edward P. Jones

<blockquote class="epigraph" style=" padding: 10px">

 [T]he pigeon had taken a step and dropped from the ledge. He caught an upwind that took him nearly as high as the tops of the empty K Street houses. He flew farther into Northeast, into the color and sounds of the city's morning. She did nothing, aside from following him, with her eyes, with her heart, as far as she could.
    
<p class ="attribution">—Edward P. Jones, "The Girl Who Raised Pigeons," <i>Lost in the City</i> (1993) </p>
    
</blockquote>

Edward P. Jones's *Lost in the City* (1993) is a collection of 14 short stories set in Washington D.C. The first short story, "The Girl Who Raised Pigeons," begins with a young girl raising homing pigeons on her roof.

How distinctive is a "pigeon" in the world of *Lost in the City*? What does this uniqueness (or lackthereof) tell us about the meaning of pigeons in first short story "The Girl Who Raised Pigeons" and the collection as a whole? These are just a few of the questions that we're going to try to answer with tf-idf.

If you already have a collection of plain text (.txt) files that you'd like to analyze, one of the easiest ways to calculate tf-idf scores is to use the Python library scikit-learn. It has a quick and nifty module called [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html), which does all the math for you behind the scenes. We will cover how to use the TfidfVectorizer in the next lesson.

In this lesson, however, we're going to calculate tf-idf scores manually because *Lost in the City* is still in-copyright, which means that, for legal reasons, we can't easily share or access plain text files of the book.

Luckily, the [HathiTrust Digital Library](https://www.hathitrust.org/)—which contains digitized books from Google Books as well as many university libraries—has released word frequencies per page for all 17 million books in its catalog. These word frequencies (plus part of speech tags) are otherwise known as "extracted features." There's a lot of text analysis that we can do with extracted features alone, including tf-idf.

So to calculate tf-idf scores for *Lost in the City*, we're going to use HathiTrust extracted features. That's why we're not using sci-kit learn's TfidfVectorizer. It works great with plain text files but not so great with extracted features.

## Breaking Down the TF-IDF Formula

But first, let's quickly discuss the tf-idf formula. The idea is pretty simple.

**tf-idf = term_frequency * inverse_document_frequency**

**term_frequency** = number of times a given term appears in document

**inverse_document_frequency** = log(total number of documents / number of documents with term) + 1**\***

You take the number of times a term occurs in a document (term frequency). Then you take the number of documents in which the same term occurs at least once divided by the total number of documents (document frequency), and you flip that fraction on its head (inverse document frequency). Then you multiply the two numbers together (term_frequency * inverse_document_frequency).

The reason we take the *inverse*, or flipped fraction, of document frequency is to boost the rarer words that occur in relatively few documents. Think about the inverse document frequency for the word "said" vs the word "pigeon." The term "said" appears in 13 (document frequency) of 14 (total documents) *Lost in the City* stories (14 / 13 --> a smaller inverse document frequency) while the term "pigeons" only occurs in 2 (document frequency) of the 14 stories (total documents) (14 / 2 --> a bigger inverse document frequency, a bigger tf-idf boost). 

*There are a bunch of slightly different ways that you can calculate inverse document frequency. The version of idf that we're going to use is the [scikit-learn default](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html#sklearn.feature_extraction.text.TfidfTransformer), which uses "smoothing" aka it adds a "1" to the numerator and denominator: 

**inverse_document_frequency**  = log((1 + total_number_of_documents) / (number_of_documents_with_term +1)) + 1

```{margin}
> If smooth_idf=True (the default), the constant “1” is added to the numerator and denominator of the idf as if an extra document was seen containing every term in the collection exactly once, which prevents zero divisions: idf(t) = log [ (1 + n) / (1 + df(t)) ] + 1.  
> -[scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html#sklearn.feature_extraction.text.TfidfTransformer)
```

### Let's test it out

We need the `log()` function for our calculation, otherwise known as [logarithm](https://en.wikipedia.org/wiki/Logarithm), so we're going to import the `numpy` package.

In [1]:
import numpy as np

**"said"**

In [2]:
total_number_of_documents = 14 ##total number of short stories in *Lost in the City*
number_of_documents_with_term = 13 ##number of short stories the contain the word "said"

In [3]:
term_frequency = 47 ##number of times "said" appears in "The Girl Who Raised Pigeons"
inverse_document_frequency = np.log((1 + total_number_of_documents) / (number_of_documents_with_term +1)) + 1

In [4]:
term_frequency * inverse_document_frequency

50.24266495988672

**"pigeons"**

In [5]:
total_number_of_documents = 14 ##total number of short stories in *Lost in the City*
number_of_documents_with_term = 2 ##number of short stories the contain the word "pigeons"

In [6]:
term_frequency = 30 ##number of times "pigeons" appears in "The Girl Who Raised Pigeons"
inverse_document_frequency = np.log((1 + total_number_of_documents) / (number_of_documents_with_term +1)) + 1

In [7]:
term_frequency * inverse_document_frequency

78.28313737302301

**tf–idf scores for "The Girl Who Raised Pigeons"**

"said" = 50.48<br>
"pigeons" = 78.28

Though the word "said" appears 47 times in "The Girl Who Raised Pigeons" and the word "pigeons" only appears 30 times, "pigeons" has a higher tf–idf score than "said" because it's a rarer word. The word "pigeons" appears in 2 of 14 stories, while "said" appears in 13 of 14 stories, almost all of them.

## Get HathiTrust Extracted Features

Now let's try to calculate tf-idf scores for all the words in all the short stories in *Lost in the City*. To do so, we need word counts, or HathiTrust extracted features, for each story in the collection.

To work with HathiTrust's extracted features, we first need to install and import the [HathiTrust Feature Reader](https://github.com/htrc/htrc-feature-reader).

Install HathiTrust Feature Reader

In [8]:
!pip install htrc-feature-reader

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 KB 979.5 kB/s eta 0:00:00 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.2/160.2 KB 3.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 8.1 MB/s eta 0:00:00a 0:00:01m
  Created wheel for htrc-feature-reader: filename=htrc_feature_reader-2.0.7-py3-none-any.whl size=40424 sha256=0bd30c8329c898b3516e3d2d483ab6c71bdcdc6b2f41f6d829f5303fd704d5fd
  Stored in directory: /home/letuvertia/.cache/pip/wheels/b3/f1/1f/5c294023ab216db36f2d0106839e71b85612d467226e0a01ee
Successfully built htrc-feature-reader


Import necessary libraries

In [9]:
from htrc_features import Volume
import pandas as pd

:::{admonition} Pandas Review
:class: pandasreview
 Do you need a refresher or introduction to the Python data analysis library Pandas? Be sure to check out <a href="https://melaniewalsh.github.io/Intro-Cultural-Analytics/Data-Analysis/Pandas-Basics-Part1.html"> Pandas Basics (1-3) </a> in this textbook!
    
:::

Then we need to locate the the HathiTrust volume ID for *Lost in the City*. If we search the HathiTrust catalog for this book and then click on "Limited (search only)," it will take us to the following web page: https://babel.hathitrust.org/cgi/pt?id=mdp.39015029970129.

The HathiTrust Volume ID for *Lost in the City* is located after `id=` this URL: `mdp.39015029970129`. 

### Make DataFrame of Word Frequencies From Volume(s)

#### Single Volume

To get HathiTrust extracted features for a single volume, we can create a [`Volume` object](https://github.com/htrc/htrc-feature-reader#volume) and use the `.tokenlist()` method. 

In [10]:
Volume('mdp.39015029970129').tokenlist()

count
page section token   pos       
1    body    ,       ,        1
             .046    CD       1
             1993    CD       1
             3560    CD       1
             AWARD   NN       1
...                         ...
260  body    world   NN       2
             would   MD       1
             writers NNS      1
             written VBN      1
             •       SYM      1

[51297 rows x 1 columns]

For each page in *Lost in the City*, this DataFrame displays the page number and section type as well as every word/token that appears on the page, its part-of-speech, and the number of times that word/token occurs on the page. As you can see, there are 51,297 rows in this DataFrame — one for each token that appears on each page.

Let's look at a sample of just 20 words from page 11.

In [11]:
Volume('mdp.39015029970129').tokenlist()[500:520]

count
page section token            pos       
11   body    out              RP       1
             over             IN       1
             part             NN       1
             past             IN       1
             pee              VB       1
             pigeon           NN       1
             pigeons          NNS      1
             reach            VB       1
             remained         VBD      1
             roof             NN       1
             room             NN       2
             say              VB       1
             seemed           VBD      1
             set              VBN      1
             share            VB       1
             she              PRP      7
             silenccBometimes NNS      1
             silence          NN       1
             simple           JJ       1
             slats            NNS      1

We can also get metadata for a HathiTrust volume by asking for [certain attributes](https://github.com/htrc/htrc-feature-reader#volume).

In [12]:
Volume('mdp.39015029970129').year

1993

In [13]:
Volume('mdp.39015029970129').page_count

260

In [14]:
Volume('mdp.39015029970129').publisher

'HarperPerennial'

#### Multiple Volumes

We might want to get extracted features for multiple volumes at the same time, so we're also going to practice a workflow that will allow us to read in multiple HathiTrust books, even though we're only reading in one book at this moment.

Insert list of desired HathiTrust volume(s)

In [15]:
volume_ids = ['mdp.39015029970129']

Loop through this list of volume IDs and make a DataFrame that includes extracted features, book title, and publication year, then make a list of all DataFrames.

In [16]:
all_tokens = []

for hathi_id in volume_ids:
    
    #Read in HathiTrust volume
    volume = Volume(hathi_id)
    
    #Make dataframe from token list -- do not include part of speech, sections, or case sensitivity
    token_df = volume.tokenlist(case=False, pos=False, drop_section=True)
    
    #Add book column
    token_df['book'] = volume.title
    
    #Add publication year column
    token_df['year'] = volume.year
    
    all_tokens.append(token_df)

Concatenate the list of DataFrames 

In [17]:
lost_df = pd.concat(all_tokens)

Preview the DataFrame

In [18]:
lost_df

count                          book  year
page lowercase                                           
1    ,              1  Lost in the city : stories /  1993
     .046           1  Lost in the city : stories /  1993
     1993           1  Lost in the city : stories /  1993
     3560           1  Lost in the city : stories /  1993
     a              1  Lost in the city : stories /  1993
...               ...                           ...   ...
260  would          1  Lost in the city : stories /  1993
     writers        1  Lost in the city : stories /  1993
     written        1  Lost in the city : stories /  1993
     york           1  Lost in the city : stories /  1993
     •              1  Lost in the city : stories /  1993

[47307 rows x 3 columns]

Change from multi-level index to regular index with `reset_index()`

In [19]:
lost_df_flattened = lost_df.reset_index()

In [20]:
lost_df_flattened 

,page,lowercase,count,book,year
0,1,",",1,Lost in the city : stories /,1993
1,1,.046,1,Lost in the city : stories /,1993
2,1,1993,1,Lost in the city : stories /,1993
3,1,3560,1,Lost in the city : stories /,1993
4,1,a,1,Lost in the city : stories /,1993
...,...,...,...,...,...
47302,260,would,1,Lost in the city : stories /,1993
47303,260,writers,1,Lost in the city : stories /,1993
47304,260,written,1,Lost in the city : stories /,1993
47305,260,york,1,Lost in the city : stories /,1993


Nice! We now have a DataFrame of word counts per page for *Lost in the City*.

But what we need to move forward with tf-idf is a way of splitting this collection into its individual stories. Remember: to use tf-idf, we need a *collection* of texts because we need to compare word frequency for one document with all the other documents in the collection.

## Add story titles

How can we split up *Lost in the City* into individual stories?

Sometimes HathiTrust Extracted Features helpfully include "section" information for a book, such as chapter titles. Unfortunately, the extracted features for *Lost in the City* do not include chapter or story titles.

They do, however, include page numbers and, if you specify `volume.tokenlist(case=True)`, words with case sensitivity. When I manually combed through the HTRC token list with case sensitivity turned on, I noticed that the title page for each short story seemed to format the title in all-caps. So I searched for all-caps words from each story title and noted down the corresponding page number. This should give us a marker of where every story begins and ends.

The function below will add in *Lost in the City*'s story titles for the correct page numbers and corresponding words.

In [22]:
def add_story_titles(page):
    if page >= 0 and page < 11:
        return "Front Matter"
    if page >= 11 and page < 35:
        return "01: The Girl Who Raised Pigeons"
    elif page >= 35 and page < 41:
        return "02: The First Day"
    elif page >= 41 and page < 63:
        return "03: The Night Rhonda Ferguson Was Killed"
    elif page >= 63 and page < 85:
        return "04: Young Lions"
    elif page >= 85 and page < 113:
        return "05: The Store"
    elif page >= 113 and page < 125:
        return "06: An Orange Line Train to Ballston"
    elif page >= 125 and page < 149:
        return "07: The Sunday Following Mother's Day"
    elif page >= 149 and page < 159:
        return "08: Lost in the City"
    elif page >= 159 and page < 184:
        return "09: His Mother's House"
    elif page >= 184 and page < 191:
        return "10: A Butterfly on F Street"
    elif page >= 191 and page < 209:
        return "11: Gospel"
    elif page >= 209 and page < 225:
        return "12: A New Man"
    elif page >= 225 and page < 237:
        return "13: A Dark Night"
    elif page >= 237 and page <= 252:
        return "14: Marie"
    elif page > 252:
        return "Back Matter"

Below we add a new column of story titles to the DataFrame by `apply()`ing our function to the "page" column and dumping the results to `lost_df_flattened['story']`. You can read more about applying functions in ["Pandas Basics - Part 3"](https://melaniewalsh.github.io/Intro-Cultural-Analytics/Data-Analysis/Pandas-Basics-Part3.html#applying-functions).

In [23]:
lost_df_flattened['story'] = lost_df_flattened['page'].apply(add_story_titles)

We're also going to drop the "Front Matter" and "Back Matter" from the DataFrame.

In [24]:
lost_df_flattened = lost_df_flattened.drop(lost_df_flattened[lost_df_flattened['story'] == 'Front Matter'].index)

In [25]:
lost_df_flattened = lost_df_flattened.drop(lost_df_flattened[lost_df_flattened['story'] == 'Back Matter'].index)

## Sum Word Counts For Each Story

Page-level information is great. But for tf-idf purposes, we really only care about the frequency of words for every story. Below we group by story and calculate the sum of word frequencies for all the pages in that story.

In [26]:
lost_df_flattened.groupby(['story', 'lowercase'])[['count']].sum().reset_index()

,story,lowercase,count
0,01: The Girl Who Raised Pigeons,!,8
1,01: The Girl Who Raised Pigeons,',4
2,01: The Girl Who Raised Pigeons,'',111
3,01: The Girl Who Raised Pigeons,'d,1
4,01: The Girl Who Raised Pigeons,'ll,5
...,...,...,...
18082,14: Marie,yet,1
18083,14: Marie,you,39
18084,14: Marie,you-know-who,1
18085,14: Marie,young,8


Notice how the "page" column no longer exists in the DataFrame and our rows have slimmed down from more than 40,000 to 18,000.

In [27]:
word_frequency_df = lost_df_flattened.groupby(['story', 'lowercase'])[['count']].sum().reset_index()

## Remove Infrequent Words, Stopwords, & Punctuation

We will conclude with some final pre-processing steps. We will remove the list of stopwords defined below.

Make list of stopwords

In [28]:
STOPS = ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 'your', 'yours',
         'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', 'her', 'hers',
         'herself', 'it', 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves',
         'what', 'which', 'who', 'whom', 'this', 'that', 'these', 'those', 'am', 'is', 'are',
         'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does',
         'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until',
         'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into',
         'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down',
         'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here',
         'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more',
         'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so',
         'than', 'too', 'very', 's', 't', 'can', 'will', 'just', 'don', 'should', 'now', 've', 'll', 'amp', "!"]

Remove stopwords

In [29]:
word_frequency_df = word_frequency_df.drop(word_frequency_df[word_frequency_df['lowercase'].isin(STOPS)].index)

We will also remove punctuation by using a regular expression `[^A-Za-z\s]`, which matches anything that's not a letter and drops it from the DataFrame.

In [30]:
word_frequency_df = word_frequency_df.drop(word_frequency_df[word_frequency_df['lowercase'].str.contains('[^A-Za-z\s]', regex=True)].index)

In [31]:
#Remove words that appear less than 5 times in a book
#word_frequency_df_test = word_frequency_df[word_frequency_df['count'] > 5]

In [32]:
word_frequency_df

,story,lowercase,count
36,01: The Girl Who Raised Pigeons,abandoned,2
37,01: The Girl Who Raised Pigeons,able,2
40,01: The Girl Who Raised Pigeons,absently,1
41,01: The Girl Who Raised Pigeons,absolute,1
42,01: The Girl Who Raised Pigeons,accepted,1
...,...,...,...
18079,14: Marie,years,10
18080,14: Marie,yes,2
18081,14: Marie,yesterday,2
18082,14: Marie,yet,1


## TF-IDF

### Term Frequency

We already have term frequencies for each document. Let's rename the columns so that they're consistent with the tf-idf vocabulary that we've been using.

In [33]:
word_frequency_df = word_frequency_df.rename(columns={'lowercase': 'term','count': 'term_frequency'})

In [34]:
word_frequency_df

,story,term,term_frequency
36,01: The Girl Who Raised Pigeons,abandoned,2
37,01: The Girl Who Raised Pigeons,able,2
40,01: The Girl Who Raised Pigeons,absently,1
41,01: The Girl Who Raised Pigeons,absolute,1
42,01: The Girl Who Raised Pigeons,accepted,1
...,...,...,...
18079,14: Marie,years,10
18080,14: Marie,yes,2
18081,14: Marie,yesterday,2
18082,14: Marie,yet,1


### Document Frequency

To calculate the number of documents or stories in which each term appears, we're going to create a separate DataFrame and do some Pandas manipulation and calculation.

In [35]:
document_frequency_df = (word_frequency_df.groupby(['story','term']).size().unstack()).sum().reset_index()

If you inspect parts of the complex chain of Pandas methods above (which is always a great way to learn!), you will see that we're momentarily reshaping the DataFrame to see if each term appears in each story...

In [36]:
word_frequency_df.groupby(['story','term']).size().unstack()

term,abandoned,abhored,abide,ability,able,abomination,aboum,aboutfcfteen,abqu,absently,...,ypu,yr,ysirs,ythe,yuddini,zigzagging,zion,zipped,zippers,zoo
story,,,,,,,,,,,,,,,,,,,,,
01: The Girl Who Raised Pigeons,1.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,1.0,...,NaN,1.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN
02: The First Day,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
03: The Night Rhonda Ferguson Was Killed,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
04: Young Lions,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,NaN,...,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
05: The Store,NaN,NaN,NaN,1.0,1.0,1.0,NaN,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
06: An Orange Line Train to Ballston,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
07: The Sunday Following Mother's Day,1.0,1.0,1.0,NaN,1.0,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
08: Lost in the City,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
09: His Mother's House,1.0,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN


Then we're adding up how many stories each term appears in (`.sum()`) and resetting the index (`.reset_index()`) to make a DataFrame.

Finally, we will rename the column in this DataFrame and merge it into our word frequency DataFrame.

In [37]:
document_frequency_df = document_frequency_df.rename(columns={0:'document_frequency'})

In [38]:
word_frequency_df = word_frequency_df.merge(document_frequency_df)

Now we have term frequency and document frequency.

In [39]:
word_frequency_df

,story,term,term_frequency,document_frequency
0,01: The Girl Who Raised Pigeons,abandoned,2,3.0
1,07: The Sunday Following Mother's Day,abandoned,1,3.0
2,09: His Mother's House,abandoned,1,3.0
3,01: The Girl Who Raised Pigeons,able,2,12.0
4,03: The Night Rhonda Ferguson Was Killed,able,3,12.0
...,...,...,...,...
15721,14: Marie,whim,1,1.0
15722,14: Marie,wilamena,20,1.0
15723,14: Marie,wise,8,1.0
15724,14: Marie,womanish,1,1.0


As you can see in the DataFrame above, the term "abandoned" appears 2 times in the story "The Girl Who Raised Pigeons" (term frequency), and it appears in 3 different stories in the collection overall (document frequency).

### Total Number of Documents 

To calculate the total number of documents are in the collection, we count how many unique values are in the "story" column (we know the answer should be 14 short stories).

In [40]:
total_number_of_documents = lost_df_flattened['story'].nunique()

In [41]:
total_number_of_documents

14

### Inverse Document Frequency

As we previously established, there are a lot of slightly different versions of the tf-idf formula, but we're going to use the default version from the [scikit-learn library](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html#sklearn.feature_extraction.text.TfidfTransformer) that adds "smoothing" to inverse document frequency.

```
inverse_document_frequency = log [ (1 + total number of docs) / (1 + document frequency) ] + 1
```

In [42]:
import numpy as np

In [43]:
word_frequency_df['idf'] = np.log((1 + total_number_of_documents) / (1 + word_frequency_df['document_frequency'])) + 1

### TF- IDF

Finally, we will calculate tf-idf by multiplying term frequency and inverse document frequency together.

In [44]:
word_frequency_df['tfidf'] = word_frequency_df['term_frequency'] * word_frequency_df['idf']

Then we will normalize these values with the scikit-learn library.

In [45]:
from sklearn import preprocessing

In [46]:
word_frequency_df['tfidf_normalized'] = preprocessing.normalize(word_frequency_df[['tfidf']], axis=0, norm='l2')

We did it! Now let's inspect the top 15 words with the highest tfidf scores for each story in the collection

In [47]:
word_frequency_df.sort_values(by=['story','tfidf_normalized'], ascending=[True,False]).groupby(['story']).head(15)

,story,term,term_frequency,document_frequency,idf,tfidf,tfidf_normalized
655,01: The Girl Who Raised Pigeons,betsy,44,1.0,3.014903,132.655733,0.106417
3317,01: The Girl Who Raised Pigeons,jenny,42,1.0,3.014903,126.625927,0.101580
212,01: The Girl Who Raised Pigeons,ann,45,2.0,2.609438,117.424706,0.094199
5566,01: The Girl Who Raised Pigeons,robert,36,1.0,3.014903,108.536509,0.087069
1384,01: The Girl Who Raised Pigeons,coop,28,1.0,3.014903,84.417285,0.067720
...,...,...,...,...,...,...,...
11578,14: Marie,social,10,2.0,2.609438,26.094379,0.020933
4769,14: Marie,one,25,14.0,1.000000,25.000000,0.020055
15552,14: Marie,calhoun,8,1.0,3.014903,24.119224,0.019349
15723,14: Marie,wise,8,1.0,3.014903,24.119224,0.019349


It turns out that "pigeons" are pretty unique to the first short story in *Lost in the City* and have a normalized tf-idf score of .062, making it one of the most distinctive words in that story along with "coop" and "birds."

What are some other distinctive words in *Lost in the City*?

## Further Resources

- Peter Organisciak and Boris Capitanu, ["Text Mining in Python through the HTRC Feature Reader,"](https://programminghistorian.org/en/lessons/text-mining-with-extracted-features) *The Programming Historian*


# 2. TF-IDF with Scikit-Learn

In the previous lesson, we learned about a text analysis method called *term frequency–inverse document frequency*, often abbreviated *tf-idf*. Tf-idf is a method that tries to identify the most distinctively frequent or significant words in a document. We specifically learned how to calculate tf-idf scores using word frequencies per page—or "extracted features"—made available by the HathiTrust Digital Library.

In this lesson, we're going to learn how to calculate tf-idf scores using a collection of plain text (.txt) files and the Python library scikit-learn, which has a quick and nifty module called [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).

In this lesson, we will cover how to:
- Calculate and normalize tf-idf scores for U.S. Inaugural Addresses with scikit-learn

## Dataset

### U.S. Inaugural Addresses

<blockquote class="epigraph" style=" padding: 10px">

This is the meaning of our liberty and our creed; why men and women and children of every race and every faith can join in celebration across this magnificent Mall, and why a man whose father less than 60 years ago might not have been served at a local restaurant can now stand before you to take a most sacred oath.  So let us mark this day with remembrance of who we are and how far we have traveled.
<p class ="attribution">—Barack Obama, Inaugural Presidential Address, January 2009
    </p>
    
</blockquote>

During Barack Obama's Inaugural Address in January 2009, he mentioned "women" four different times, including in the passage quoted above. How distinctive is Obama's inclusion of women in this address compared to all other U.S. Presidents? This is one of the questions that we're going to try to answer with tf-idf.

## Breaking Down the TF-IDF Formula

But first, let's quickly discuss the tf-idf formula. The idea is pretty simple.

**tf-idf = term_frequency * inverse_document_frequency**

**term_frequency** = number of times a given term appears in document

**inverse_document_frequency** = log(total number of documents / number of documents with term) + 1**\***

You take the number of times a term occurs in a document (term frequency). Then you take the number of documents in which the same term occurs at least once divided by the total number of documents (document frequency), and you flip that fraction on its head (inverse document frequency). Then you multiply the two numbers together (term_frequency * inverse_document_frequency).

The reason we take the *inverse*, or flipped fraction, of document frequency is to boost the rarer words that occur in relatively few documents. Think about the inverse document frequency for the word "said" vs the word "pigeon." The term "said" appears in 13 (document frequency) of 14 (total documents) *Lost in the City* stories (14 / 13 --> a smaller inverse document frequency) while the term "pigeons" only occurs in 2 (document frequency) of the 14 stories (total documents) (14 / 2 --> a bigger inverse document frequency, a bigger tf-idf boost). 

*There are a bunch of slightly different ways that you can calculate inverse document frequency. The version of idf that we're going to use is the [scikit-learn default](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html#sklearn.feature_extraction.text.TfidfTransformer), which uses "smoothing" aka it adds a "1" to the numerator and denominator: 

**inverse_document_frequency**  = log((1 + total_number_of_documents) / (number_of_documents_with_term +1)) + 1

<div class="margin sidebar" style=" padding: 10px">

> If smooth_idf=True (the default), the constant “1” is added to the numerator and denominator of the idf as if an extra document was seen containing every term in the collection exactly once, which prevents zero divisions: idf(t) = log [ (1 + n) / (1 + df(t)) ] + 1.  
> -[scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html#sklearn.feature_extraction.text.TfidfTransformer)

</div>

## TF-IDF with scikit-learn

[scikit-learn](https://scikit-learn.org/stable/index.html), imported as `sklearn`, is a popular Python library for machine learning approaches such as clustering, classification, and regression. Though we're not doing any machine learning in this lesson, we're nevertheless going to use scikit-learn's `TfidfVectorizer` and `CountVectorizer`.

Install scikit-learn

In [50]:
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable


Import necessary modules and libraries

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
pd.set_option('display.max_rows', 100)
from pathlib import Path  
import glob

We're also going to import `pandas` and change its default display setting. And we're going to import two libraries that will help us work with files and the file system: [`pathlib`](https://docs.python.org/3/library/pathlib.html##basic-use) and [`glob`](https://docs.python.org/3/library/glob.html).

#### Set Directory Path

Below we're setting the directory filepath that contains all the text files that we want to analyze.

In [53]:
directory_path = "../texts/history/US_Inaugural_Addresses/"

Then we're going to use `glob` and `Path` to make a list of all the filepaths in that directory and a list of all the short story titles.

In [54]:
text_files = glob.glob(f"{directory_path}/*.txt")

In [55]:
text_files

['../texts/history/US_Inaugural_Addresses/01_washington_1789.txt',
 '../texts/history/US_Inaugural_Addresses/02_washington_1793.txt',
 '../texts/history/US_Inaugural_Addresses/03_adams_john_1797.txt',
 '../texts/history/US_Inaugural_Addresses/04_jefferson_1801.txt',
 '../texts/history/US_Inaugural_Addresses/05_jefferson_1805.txt',
 '../texts/history/US_Inaugural_Addresses/06_madison_1809.txt',
 '../texts/history/US_Inaugural_Addresses/07_madison_1813.txt',
 '../texts/history/US_Inaugural_Addresses/08_monroe_1817.txt',
 '../texts/history/US_Inaugural_Addresses/09_monroe_1821.txt',
 '../texts/history/US_Inaugural_Addresses/10_adams_john_quincy_1825.txt',
 '../texts/history/US_Inaugural_Addresses/11_jackson_1829.txt',
 '../texts/history/US_Inaugural_Addresses/12_jackson_1833.txt',
 '../texts/history/US_Inaugural_Addresses/13_van_buren_1837.txt',
 '../texts/history/US_Inaugural_Addresses/14_harrison_1841.txt',
 '../texts/history/US_Inaugural_Addresses/15_polk_1845.txt',
 '../texts/history/

In [56]:
text_titles = [Path(text).stem for text in text_files]

In [57]:
text_titles

['01_washington_1789',
 '02_washington_1793',
 '03_adams_john_1797',
 '04_jefferson_1801',
 '05_jefferson_1805',
 '06_madison_1809',
 '07_madison_1813',
 '08_monroe_1817',
 '09_monroe_1821',
 '10_adams_john_quincy_1825',
 '11_jackson_1829',
 '12_jackson_1833',
 '13_van_buren_1837',
 '14_harrison_1841',
 '15_polk_1845',
 '16_taylor_1849',
 '17_pierce_1853',
 '18_buchanan_1857',
 '19_lincoln_1861',
 '20_lincoln_1865',
 '21_grant_1869',
 '22_grant_1873',
 '23_hayes_1877',
 '24_garfield_1881',
 '25_cleveland_1885',
 '26_harrison_1889',
 '27_cleveland_1893',
 '28_mckinley_1897',
 '29_mckinley_1901',
 '30_roosevelt_theodore_1905',
 '31_taft_1909',
 '32_wilson_1913',
 '33_wilson_1917',
 '34_harding_1921',
 '35_coolidge_1925',
 '36_hoover_1929',
 '37_roosevelt_franklin_1933',
 '38_roosevelt_franklin_1937',
 '39_roosevelt_franklin_1941',
 '40_roosevelt_franklin_1945',
 '41_truman_1949',
 '42_eisenhower_1953',
 '43_eisenhower_1957',
 '44_kennedy_1961',
 '45_johnson_1965',
 '46_nixon_1969',
 '47_

## Calculate tf–idf

To calculate tf–idf scores for every word, we're going to use scikit-learn's [`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html).

When you initialize TfidfVectorizer, you can choose to set it with different parameters. These parameters will change the way you calculate tf–idf.

The recommended way to run `TfidfVectorizer` is with smoothing (`smooth_idf = True`) and normalization (`norm='l2'`) turned on. These parameters will better account for differences in text length, and overall produce more meaningful tf–idf scores. Smoothing and L2 normalization are actually the default settings for `TfidfVectorizer`, so to turn them on, you don't need to include any extra code at all.

Initialize TfidfVectorizer with desired parameters (default smoothing and normalization)

In [58]:
tfidf_vectorizer = TfidfVectorizer(input='filename', stop_words='english')

Run TfidfVectorizer on our `text_files`

In [59]:
tfidf_vector = tfidf_vectorizer.fit_transform(text_files)

Make a DataFrame out of the resulting tf–idf vector, setting the "feature names" or words as columns and the titles as rows

In [60]:
tfidf_df = pd.DataFrame(tfidf_vector.toarray(), index=text_titles, columns=tfidf_vectorizer.get_feature_names_out())

Add column for document frequency aka number of times word appears in all documents

In [61]:
tfidf_df.loc['00_Document Frequency'] = (tfidf_df > 0).sum()

In [62]:
tfidf_slice = tfidf_df[['government', 'borders', 'people', 'obama', 'war', 'honor','foreign', 'men', 'women', 'children']]
tfidf_slice.sort_index().round(decimals=2)

,government,borders,people,obama,war,honor,foreign,men,women,children
00_Document Frequency,53.00,5.00,56.00,3.00,45.00,32.00,32.00,47.00,15.00,22.00
01_washington_1789,0.11,0.00,0.05,0.00,0.00,0.00,0.00,0.02,0.00,0.00
02_washington_1793,0.06,0.00,0.05,0.00,0.00,0.08,0.00,0.00,0.00,0.00
03_adams_john_1797,0.16,0.00,0.19,0.00,0.01,0.10,0.12,0.04,0.00,0.00
04_jefferson_1801,0.16,0.00,0.01,0.00,0.01,0.04,0.00,0.04,0.00,0.00
05_jefferson_1805,0.03,0.00,0.00,0.00,0.04,0.00,0.06,0.01,0.00,0.02
06_madison_1809,0.00,0.00,0.02,0.00,0.02,0.05,0.05,0.00,0.00,0.00
07_madison_1813,0.04,0.00,0.04,0.00,0.25,0.02,0.02,0.00,0.00,0.00
08_monroe_1817,0.17,0.00,0.11,0.00,0.09,0.01,0.10,0.04,0.00,0.00
09_monroe_1821,0.08,0.00,0.06,0.00,0.11,0.02,0.04,0.01,0.00,0.01


Let's drop "OO_Document Frequency" since we were just using it for illustration purposes.

In [63]:
tfidf_df = tfidf_df.drop('00_Document Frequency', errors='ignore')

Let's reorganize the DataFrame so that the words are in rows rather than columns.

In [64]:
tfidf_df.stack().reset_index()

,level_0,level_1,0
0,01_washington_1789,000,0.000000
1,01_washington_1789,03,0.000000
2,01_washington_1789,04,0.023259
3,01_washington_1789,05,0.000000
4,01_washington_1789,100,0.000000
...,...,...,...
521937,58_trump_2017,zachary,0.000000
521938,58_trump_2017,zeal,0.000000
521939,58_trump_2017,zealous,0.000000
521940,58_trump_2017,zealously,0.000000


In [65]:
tfidf_df = tfidf_df.stack().reset_index()

In [66]:
tfidf_df = tfidf_df.rename(columns={0:'tfidf', 'level_0': 'document','level_1': 'term', 'level_2': 'term'})

To find out the top 10 words with the highest tf–idf for every story, we're going to sort by document and tfidf score and then groupby document and take the first 10 values.

In [67]:
tfidf_df.sort_values(by=['document','tfidf'], ascending=[True,False]).groupby(['document']).head(10)

,document,term,tfidf
3707,01_washington_1789,government,0.113681
4108,01_washington_1789,immutable,0.103883
4175,01_washington_1789,impressions,0.103883
6337,01_washington_1789,providential,0.103883
5631,01_washington_1789,ought,0.103728
...,...,...,...
518409,58_trump_2017,obama,0.120288
518766,58_trump_2017,people,0.112370
521001,58_trump_2017,thank,0.109171
513989,58_trump_2017,borders,0.107075


In [68]:
top_tfidf = tfidf_df.sort_values(by=['document','tfidf'], ascending=[True,False]).groupby(['document']).head(10)

We can zoom in on particular words and particular documents.

In [69]:
top_tfidf[top_tfidf['term'].str.contains('women')]

,document,term,tfidf
503861,56_obama_2009,women,0.084859


It turns out that the term "women" is very distinctive in Obama's Inaugural Address.

In [70]:
top_tfidf[top_tfidf['document'].str.contains('obama')]

,document,term,tfidf
495406,56_obama_2009,america,0.148351
500298,56_obama_2009,nation,0.120229
500358,56_obama_2009,new,0.118002
503093,56_obama_2009,today,0.114792
498590,56_obama_2009,generation,0.100654
499762,56_obama_2009,let,0.091100
499578,56_obama_2009,jobs,0.090727
496911,56_obama_2009,crisis,0.087235
498779,56_obama_2009,hard,0.084859
503861,56_obama_2009,women,0.084859


In [71]:
top_tfidf[top_tfidf['document'].str.contains('trump')]

,document,term,tfidf
513404,58_trump_2017,america,0.350162
515585,58_trump_2017,dreams,0.156436
513405,58_trump_2017,american,0.149226
517576,58_trump_2017,jobs,0.142766
519262,58_trump_2017,protected,0.132439
518409,58_trump_2017,obama,0.120288
518766,58_trump_2017,people,0.112370
521001,58_trump_2017,thank,0.109171
513989,58_trump_2017,borders,0.107075
521596,58_trump_2017,ve,0.107075


In [72]:
top_tfidf[top_tfidf['document'].str.contains('kennedy')]

,document,term,tfidf
391774,44_kennedy_1961,let,0.267869
394306,44_kennedy_1961,sides,0.262849
392921,44_kennedy_1961,pledge,0.160960
387632,44_kennedy_1961,ask,0.107713
387864,44_kennedy_1961,begin,0.106495
388991,44_kennedy_1961,dare,0.106495
395895,44_kennedy_1961,world,0.103110
390313,44_kennedy_1961,final,0.102311
392370,44_kennedy_1961,new,0.096600
390120,44_kennedy_1961,explore,0.094223


## Visualize TF-IDF

We can also visualize our TF-IDF results with the data visualization library Altair.

In [73]:
!pip install altair

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.2/731.2 KB 4.2 MB/s eta 0:00:00a 0:00:01


Let's make a heatmap that shows the highest TF-IDF scoring words for each president, and let's put a red dot next to two terms of interest: "war" and "peace":

The code below was contributed by [Eric Monson](https://github.com/emonson). Thanks, Eric!

In [74]:
import altair as alt
import numpy as np

# Terms in this list will get a red dot in the visualization
term_list = ['war', 'peace']

# adding a little randomness to break ties in term ranking
top_tfidf_plusRand = top_tfidf.copy()
top_tfidf_plusRand['tfidf'] = top_tfidf_plusRand['tfidf'] + np.random.rand(top_tfidf.shape[0])*0.0001

# base for all visualizations, with rank calculation
base = alt.Chart(top_tfidf_plusRand).encode(
    x = 'rank:O',
    y = 'document:N'
).transform_window(
    rank = "rank()",
    sort = [alt.SortField("tfidf", order="descending")],
    groupby = ["document"],
)

# heatmap specification
heatmap = base.mark_rect().encode(
    color = 'tfidf:Q'
)

# red circle over terms in above list
circle = base.mark_circle(size=100).encode(
    color = alt.condition(
        alt.FieldOneOfPredicate(field='term', oneOf=term_list),
        alt.value('red'),
        alt.value('#FFFFFF00')        
    )
)

# text labels, white for darker heatmap colors
text = base.mark_text(baseline='middle').encode(
    text = 'term:N',
    color = alt.condition(alt.datum.tfidf >= 0.23, alt.value('white'), alt.value('black'))
)

# display the three superimposed visualizations
(heatmap + circle + text).properties(width = 600)

alt.LayerChart(...)

## Your Turn!

Take a few minutes to explore the dataframe below and then answer the following questions.

**1.** What is the difference between a tf-idf score and raw word frequency?

**Your answer here**

**2.** Based on the dataframe above, what is one potential problem or limitation that you notice with tf-idf scores?

**Your answer here**

**3.** What's another collection of texts that you think might be interesting to analyze with tf-idf scores?  Why?

**Your answer here**